# CS229 Simplified PS2 — Classification and Neural Networks

参考 CS229 logistic stability、imbalanced labels 与 simple-NN 作业题型。

这不是 Stanford 官方作业副本，而是用于自学的**简化原创版本**。题型参考公开的
CS229 problem sets；公式以 2026 Main Notes 为准，数据由本仓库确定性生成。

- [Spring 2026 课程主页](https://cs229.stanford.edu/index.html-spr26)
- [Spring 2026 公开视频](https://www.youtube.com/playlist?list=PLaqpC4kq8Gpw)
- [CS229 2026 Main Notes](https://cs229.stanford.edu/main_notes.pdf)
- [Stanford 官方公开的 Summer 2020 problem sets](https://cs229.stanford.edu/summer2020/)
- [公开可见的 Summer 2025 作业结构参考](https://github.com/MDzimah/Stanford-University-CS229-Machine-Learning)

**使用规则：** 先手推，再写代码。不要先看学生 solution。每道题完成后才把对应的
`RUN_*_CHECKS` 改成 `True`。检查不通过时，从 shape 和公式开始定位。

**怎么读题：** `m` 表示样本数，`d` 表示设计矩阵的列数（含截距列时也计入）。
`X` 的每一行是一条样本，`y[i]` 是对应目标；`(m,)` 是一维数组，和 `(m, 1)` 不同。
“实现”指填写函数中的 TODO；“运行检查”指执行已经给出的调用代码；“解释”写在答题框中。
函数定义格和检查格需要分别运行。检查通过只代表已检查的条件成立，不代替解释实验结果。

## Goal

对应 Spring 2026 Lecture 3–8。先在 logistic regression 的语境里正式引入 binary
classifier 和 cross-entropy，再研究类别不平衡，最后实现一个两层神经网络。

## Setup

`label` 为 0 或 1。正类样本较少，这是题目故意设置的实验条件。

加载函数已给每行加入截距：`X_train` 为 `(200, 3)`，每行为 `[1, x1, x2]`，
`y_train` 为 `(200,)`；验证集 100 行。不要再添加一列 1。
训练集用于拟合，验证集用于比较；所有模型都用概率阈值 0.5 生成类别。
P2 需要先运行 P1 检查格得到预测，P3 复用 P1–P2 函数；P4 使用另外给出的 XOR 数据。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(229)

candidates = [Path("data"), Path("problem_sets/ps2_classification/data")]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Run: python scripts/build_problem_sets.py")

print("Data directory:", DATA_DIR.resolve())


### 加载并查看本题集的数据

上格准备了库和数据目录。下格读取数据、打印 shape，并展示实验输入；按顺序运行即可。

In [ ]:
def load_classification(filename):
    data = np.loadtxt(DATA_DIR / filename, delimiter=",", skiprows=1)
    X = data[:, :2]
    y = data[:, 2].astype(int)
    X = np.column_stack([np.ones(len(X)), X])
    return X, y

X_train, y_train = load_classification("train.csv")
X_valid, y_valid = load_classification("valid.csv")
print("train:", X_train.shape, y_train.shape, "positive rate:", y_train.mean())
print("valid:", X_valid.shape, y_valid.shape, "positive rate:", y_valid.mean())


## Problem 1 — Logistic regression from scratch

**本题目的：** 从训练数据学习一个二分类模型：输入特征，输出属于类别 1 的概率。先写概率转换，再写损失，最后写训练循环。

先在答题框推导平均二元交叉熵的梯度。公式在下方损失函数旁，推导时先取所有样本权重为 1。

> **你的答案（请双击本格后填写）**
>
> TODO：推导 binary cross-entropy gradient，并写出每个量的 shape。

### 1. sigmoid

**要写什么：** `sigmoid(z)` 将模型分数变成 0 到 1 的概率，输入输出 shape 相同。

$$\sigma(z)=\frac{1}{1+e^{-z}}.$$

`z=0` 时结果为 `0.5`。请处理 `[-1000.,0.,1000.]` 这样的极端输入，
输出应接近 `[0.,0.5,1.]` 且没有 NaN；可按正负数分支避免指数溢出。

In [ ]:
def sigmoid(z):
    # START TODO
    raise NotImplementedError("Implement a numerically stable sigmoid")


**只检查刚才这个函数：** 填完并运行函数定义后，把下格 `TRY_SIGMOID` 改为 `True`。
先用这个小例子核对输入输出，再继续下一步。

In [ ]:
TRY_SIGMOID = False
if TRY_SIGMOID:
    assert np.allclose(sigmoid(np.array([-1000., 0., 1000.])), [0., 0.5, 1.])
    print("sigmoid: example passed")


### 2. 准备数据

先运行这格准备本题数据，后面的函数将使用它。

In [ ]:
# END TODO


### 3. binary_cross_entropy

**要写什么：** `binary_cross_entropy(y, probability, sample_weight=None)` 衡量概率预测好坏。
`y` 和 `probability` 都为 `(m,)`，返回一个损失值。这里使用自然对数。

$$L_w=-\frac{\sum_i w_i[y_i\log p_i+(1-y_i)\log(1-p_i)]}{\sum_i w_i}.$$

`None` 表示所有权重为 1，即普通平均损失。传入的权重为非负 `(m,)`，总和必须大于 0。
取对数前将概率裁剪到 `[1e-12,1-1e-12]`。例如真实标签 1 的预测概率为 0.9，
比预测概率 0.1 的损失小。P3 会复用这个函数来改变样本权重。

In [ ]:
def binary_cross_entropy(y, probability, sample_weight=None):
    # START TODO
    raise NotImplementedError("Clip probabilities and return weighted mean loss")


**只检查刚才这个函数：** 填完并运行函数定义后，把下格 `TRY_BINARY_CROSS_ENTROPY` 改为 `True`。
先用这个小例子核对输入输出，再继续下一步。

In [ ]:
TRY_BINARY_CROSS_ENTROPY = False
if TRY_BINARY_CROSS_ENTROPY:
    assert binary_cross_entropy(np.array([1.]), np.array([0.9])) < binary_cross_entropy(np.array([1.]), np.array([0.1]))
    print("binary_cross_entropy: example passed")


### 4. 准备数据

先运行这格准备本题数据，后面的函数将使用它。

In [ ]:
# END TODO


### 5. fit_logistic_regression

**要写什么：** `fit_logistic_regression` 将刚才两个函数串成训练循环。
`X:(m,d)` 已有截距列，`y:(m,)` 是 0/1 标签。返回参数 `(d,)` 和长度为 `steps` 的 loss 数组。

从零参数开始，每轮算 `X @ theta` → 用 `sigmoid` 转成概率 → 记录更新前交叉熵
→ 按上方推导的梯度更新参数。`sample_weight=None` 等于全 1 权重；
有权重时梯度也要加权，且除以权重总和，不能只改 loss。
先验证全 1 权重与 None 等价。概率转类别的阈值 0.5 留到后面的完整实验使用。

In [ ]:
def fit_logistic_regression(X, y, sample_weight=None, learning_rate=0.1, steps=2000):
    theta = np.zeros(X.shape[1])
    history = []
    for step in range(steps):
        # START TODO
        raise NotImplementedError("Compute probabilities, gradient, and update")
        # END TODO
    return theta, np.array(history)


### 6. 串起来：完整实验

先测概率和损失的边界情况，再训练 theta_log；用验证输入算 probability，最后以 0.5 为阈值得到 prediction。这两个变量会供下一题使用。

完成相关函数后，打开下格的 `RUN_P1_CHECKS` 并运行。

In [ ]:
RUN_P1_CHECKS = False
if RUN_P1_CHECKS:
    extreme = sigmoid(np.array([-1000., 0., 1000.]))
    assert np.all(np.isfinite(extreme))
    assert np.allclose(extreme, [0., 0.5, 1.])
    assert np.isfinite(binary_cross_entropy(np.array([1., 0.]), np.array([0., 1.])))
    theta_log, history = fit_logistic_regression(X_train, y_train)
    probability = sigmoid(X_valid @ theta_log)
    prediction = (probability >= 0.5).astype(int)
    print("theta:", theta_log)
    print("first probabilities:", probability[:5])
    assert theta_log.shape == (3,)
    assert np.all((probability >= 0) & (probability <= 1))
    assert history[-1] < history[0]
else:
    print("P1 checks are off.")


## Problem 2 — 为什么 accuracy 会骗人

**本题目的：** 检验一个看似准确的模型有没有漏掉少数类。实现指标后，在同一验证集比较全猜 0 的模型与 P1 模型；先运行 P1 的完整实验。

### 1. classification_metrics

**要写什么：** `classification_metrics(y_true,y_pred)` 接收两个 `(m,)` 的 0/1 数组，
返回下面这些键组成的字典。`y_pred` 是类别，不是概率；约定 1 为正类。

| 键 | 如何计算 |
|---|---|
| `TP` / `TN` | 真实、预测同为 1 / 同为 0 的数量 |
| `FP` / `FN` | 真实 0 预测 1 / 真实 1 预测 0 的数量 |
| `accuracy` | `(TP+TN)/m` |
| `precision` | `TP/(TP+FP)`，预测为正的样本里多少是真的 |
| `recall` | `TP/(TP+FN)`，真实正类里找到了多少 |
| `specificity` | `TN/(TN+FP)`，真实负类里判对了多少 |
| `balanced_accuracy` | `(recall+specificity)/2` |
| `f1` | `2*precision*recall/(precision+recall)` |

分母为 0 时本练习约定该指标返回 `0.0`。
小例子：真实 `[0,0,1,1]`、预测 `[0,1,0,1]`，四种计数各为 1。

In [ ]:
def classification_metrics(y_true, y_pred):
    # Return a dictionary containing counts and requested metrics.
    # START TODO
    raise NotImplementedError("Compute confusion counts and metrics")
    # END TODO


**只检查刚才这个函数：** 填完并运行函数定义后，把下格 `TRY_CLASSIFICATION_METRICS` 改为 `True`。
先用这个小例子核对输入输出，再继续下一步。

In [ ]:
TRY_CLASSIFICATION_METRICS = False
if TRY_CLASSIFICATION_METRICS:
    example = classification_metrics(np.array([0, 0, 1, 1]), np.array([0, 1, 0, 1]))
    assert all(example[key] == 1 for key in ("TP", "TN", "FP", "FN"))
    print("classification_metrics: example passed")


### 2. 串起来：完整实验

先用四个样本核对计数，再分别计算全零预测与 P1 prediction 的指标。引用实际 accuracy、recall、balanced_accuracy 解释差别。

完成相关函数后，打开下格的 `RUN_P2_CHECKS` 并运行。

In [ ]:
RUN_P2_CHECKS = False
if RUN_P2_CHECKS:
    example = classification_metrics(np.array([0, 0, 1, 1]), np.array([0, 1, 0, 1]))
    assert all(example[key] == 1 for key in ("TP", "TN", "FP", "FN"))
    assert np.isclose(example["balanced_accuracy"], 0.5)
    all_zero = np.zeros_like(y_valid)
    print("all-zero baseline:", classification_metrics(y_valid, all_zero))
    print("logistic model  :", classification_metrics(y_valid, prediction))
else:
    print("P2 checks are off. Complete and run P1 first.")


> **你的答案（请双击本格后填写）**
>
> TODO：引用你算出的数字，解释为什么 stakeholder 不能只看 accuracy。

## Problem 3 — 给少数类更高权重

**本题目的：** 让训练过程更重视少数类，观察哪些预测变好了、哪些变差了。复用前两题的函数；验证数据保持不变。

### 1. 串起来：完整实验

用训练标签统计 n0、n1 → 给正类权重 n0/n1、负类权重 1 → 用加权函数从零训练 → 在原验证集预测 → 按同一 0.5 阈值比较指标。

下格仍需要你填写实验代码：保持与普通模型相同的学习率和更新次数。
比较 recall、specificity、balanced_accuracy、precision，在代码注释里记录哪些错误变多或变少。

完成相关函数后，打开下格的 `RUN_P3_CHECKS` 并运行。

In [ ]:
RUN_P3_CHECKS = False
if RUN_P3_CHECKS:
    # START TODO
    raise NotImplementedError("Create weights, refit, and compare metrics")
    # END TODO
else:
    print("P3 checks are off.")


## Problem 4 — 两层神经网络处理 XOR

**本题目的：** 让两层神经网络学会 XOR：两个输入不同则输出 1，相同则输出 0。分别实现前向预测和反向求梯度，再用给定循环训练。这里只检查四个训练点，不评价泛化。

### 1. 准备数据

先运行这格准备本题数据，后面的函数将使用它。每行是 XOR 的两个输入，`y_xor` 是相应的 0/1 标签；输入不加截距列。

In [ ]:
X_xor = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = np.array([0., 1., 1., 0.])


### 2. init_mlp

**已给出：** `init_mlp(rng)` 创建网络的初始参数字典，运行即可。
`W1:(2,4)` 把两个输入送到四个隐藏单元；`W2:(4,1)` 再合成一个输出。
`b1:(4,)` 和 `b2:(1,)` 是偏置，不需要给 XOR 输入加截距列。

In [ ]:
def init_mlp(rng):
    return {
        "W1": rng.normal(scale=0.5, size=(2, 4)),
        "b1": np.zeros(4),
        "W2": rng.normal(scale=0.5, size=(4, 1)),
        "b2": np.zeros(1),
    }


### 3. mlp_forward

**要写什么：** `mlp_forward(X,params)` 用当前网络参数计算概率。

$$Z_1=XW_1+b_1,\quad H=\tanh(Z_1),\quad Z_2=HW_2+b_2,\quad P=\sigma(Z_2).$$

输入 `X:(m,2)` 和参数字典，返回 `(probability,cache)`。
`probability` 必须是 `(m,1)`，`cache` 用字典保存 `H` 和 `probability`，
供下一步反向传播复用。先在注释里写每个中间量的 shape，再写运算。

In [ ]:
def mlp_forward(X, params):
    # START TODO
    raise NotImplementedError("Return probability and a cache for backprop")


### 4. 准备数据

先运行这格准备本题数据，后面的函数将使用它。每行是 XOR 的两个输入，`y_xor` 是相应的 0/1 标签；输入不加截距列。

In [ ]:
# END TODO


### 5. mlp_backward

**要写什么：** `mlp_backward(X,y,params,cache)` 计算平均交叉熵对每个网络参数的梯度。
它返回含 `W1,b1,W2,b2` 的字典，各梯度 shape 与对应参数相同；这里不更新参数。

先将 `y:(m,)` 变为 `(m,1)` 再与概率相减，否则会广播出错误的 `(m,m)`。
从输出往隐藏层使用链式法则；`tanh` 的导数可写为 `1-H**2`。
按样本数归一化，对偏置沿样本轴求和。在代码注释中记录各中间量 shape。

In [ ]:
def mlp_backward(X, y, params, cache):
    # START TODO
    raise NotImplementedError("Return gradients matching every parameter shape")


### 6. 准备数据

先运行这格准备本题数据，后面的函数将使用它。每行是 XOR 的两个输入，`y_xor` 是相应的 0/1 标签；输入不加截距列。

In [ ]:
# END TODO


### 7. 串起来：完整实验

初始化参数 → 前向得到概率和 cache → 反向得到各梯度 → 更新各参数，重复 5000 次。最后检查四个 XOR 标签；这是训练拟合检查。

完成相关函数后，打开下格的 `RUN_P4_CHECKS` 并运行。

In [ ]:
RUN_P4_CHECKS = False
if RUN_P4_CHECKS:
    params = init_mlp(np.random.default_rng(229))
    for step in range(5000):
        probability, cache = mlp_forward(X_xor, params)
        grads = mlp_backward(X_xor, y_xor, params, cache)
        for name in params:
            assert grads[name].shape == params[name].shape
            params[name] -= 0.1 * grads[name]
    probability, _ = mlp_forward(X_xor, params)
    print("XOR probabilities:", probability.ravel())
    assert np.mean((probability.ravel() >= 0.5) == y_xor) == 1.0
else:
    print("P4 checks are off.")


## Checks

- [ ] 我能解释 classifier 为什么从 Lecture 3 才出现。
- [ ] sigmoid 和 cross-entropy 在极端输入下不产生 NaN。
- [ ] 我报告了 minority-class 指标，而不只报告 accuracy。
- [ ] backprop 返回的每个梯度与对应参数 shape 相同。

## Next Steps

完成后进入 PS3，处理没有标签或带隐变量的数据。